In [2]:
from transformers import DetrImageProcessor, DetrForObjectDetection

detr_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
detr_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
detr_model.to(DEVICE)
detr_model.eval()

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NameError: name 'torch' is not defined

## Vit+DETR

In [3]:
import cv2
import torch
import numpy as np
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    DetrImageProcessor,
    DetrForObjectDetection
)
from PIL import Image
from IPython.display import Video, display
import gc

# ---------------- CONFIG ----------------
FINETUNED_MODEL_PATH = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Model/vit_final_model"
VIDEO_PATH = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Data/Video/EDA/Weapon Used Video/Battle_Of_Baghpat_2.0_Kulhads_Fly_As_Mathura_Lassi_Sellers_Fight_On_Street_360P.mp4"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FRAME_INTERVAL = 30
DETR_THRESHOLD = 0.7
# ---------------------------------------

print("🖥️ Using device:", DEVICE)

# ---------------- LOAD ViT ----------------
vit_processor = AutoImageProcessor.from_pretrained(FINETUNED_MODEL_PATH)
vit_model = AutoModelForImageClassification.from_pretrained(FINETUNED_MODEL_PATH)
vit_model.to(DEVICE)
vit_model.eval()

id2label = vit_model.config.id2label
print("🔖 ViT Labels:", id2label)

# ---------------- LOAD DETR ----------------
detr_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
detr_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
detr_model.to(DEVICE)
detr_model.eval()

print("🔍 DETR Loaded Successfully")

# ---------------- STORAGE ----------------
prob_storage = {label: [] for label in id2label.values()}
all_detected_objects = []

# Show Video
print("\n🎬 TEST VIDEO PREVIEW")
display(Video(VIDEO_PATH, embed=True))

# ---------------- VIDEO PROCESSING ----------------
cap = cv2.VideoCapture(VIDEO_PATH)
frame_count = 0

with torch.no_grad():
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % FRAME_INTERVAL == 0:

            # Convert BGR → RGB → PIL
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame_rgb)

            # =========================
            # 1️⃣ ViT CLASSIFICATION
            # =========================
            vit_inputs = vit_processor(image, return_tensors="pt")
            vit_inputs = {k: v.to(DEVICE) for k, v in vit_inputs.items()}

            vit_outputs = vit_model(**vit_inputs)
            probs = torch.softmax(vit_outputs.logits, dim=1)[0].cpu().numpy()

            for idx, prob in enumerate(probs):
                label = id2label[idx]
                prob_storage[label].append(prob)

            # =========================
            # 2️⃣ DETR OBJECT DETECTION
            # =========================
            detr_inputs = detr_processor(images=image, return_tensors="pt")
            detr_inputs = {k: v.to(DEVICE) for k, v in detr_inputs.items()}

            detr_outputs = detr_model(**detr_inputs)

            target_sizes = torch.tensor([image.size[::-1]]).to(DEVICE)
            results = detr_processor.post_process_object_detection(
                detr_outputs,
                target_sizes=target_sizes,
                threshold=DETR_THRESHOLD
            )[0]

            for score, label, box in zip(
                results["scores"],
                results["labels"],
                results["boxes"]
            ):
                class_name = detr_model.config.id2label[label.item()]
                all_detected_objects.append(class_name)

        frame_count += 1

cap.release()

# Clear memory
gc.collect()
torch.cuda.empty_cache()

# ---------------- FINAL RESULTS ----------------

# Average probabilities
avg_probs = {
    label: (np.mean(values) * 100 if values else 0.0)
    for label, values in prob_storage.items()
}

final_vit_label = max(avg_probs, key=avg_probs.get)
violence_prob = avg_probs.get("Violence", 0)

# Remove duplicates from detected objects
unique_objects = list(set(all_detected_objects))

# Define weapon list (COCO supported)
weapon_list = ["knife", "baseball bat", "sports ball"]
weapon_detected = any(obj in weapon_list for obj in unique_objects)

# ---------------- FUSION DECISION ----------------
if violence_prob > 60 and weapon_detected:
    final_decision = "🔥 HIGH CONFIDENCE VIOLENCE (Weapon Detected)"
elif violence_prob > 60:
    final_decision = "⚠ Violence Detected (No Weapon Found)"
elif weapon_detected:
    final_decision = "🟡 Weapon Present - Monitor Situation"
else:
    final_decision = "✅ Non-Violence"

# ---------------- PRINT RESULTS ----------------

print("\n🎥 VIDEO ANALYSIS RESULT")
for label, prob in avg_probs.items():
    print(f"{label:15}: {prob:.2f}%")

print("\n🔎 Detected Objects:", unique_objects)
print(f"\n🏁 FINAL DECISION: {final_decision}")

🖥️ Using device: cuda


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

🔖 ViT Labels: {0: 'NonViolence', 1: 'Violence'}


Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔍 DETR Loaded Successfully

🎬 TEST VIDEO PREVIEW



🎥 VIDEO ANALYSIS RESULT
NonViolence    : 11.07%
Violence       : 88.93%

🔎 Detected Objects: ['person', 'suitcase', 'dining table', 'chair', 'umbrella', 'cow', 'hot dog']

🏁 FINAL DECISION: ⚠ Violence Detected (No Weapon Found)


## DETR

In [ ]:
import cv2
import torch
import numpy as np
from transformers import DetrImageProcessor, DetrForObjectDetection
from PIL import Image
from IPython.display import Video, display
import gc

# ---------------- CONFIG ----------------
VIDEO_PATH = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Data/Video/EDA/Weapon Used Video/Mother of heavy weapons🥵🔥#mmg#nsg#army#indianarmy#proudindian#nsgcommando#blackcats#specialforces.mp4"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FRAME_INTERVAL =10
DETR_THRESHOLD = 0.4
MIN_WEAPON_FRAMES = 3   # Minimum frames weapon must appear
# ---------------------------------------

print("🖥️ Using device:", DEVICE)

# ---------------- LOAD DETR ----------------
detr_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
detr_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
detr_model.to(DEVICE)
detr_model.eval()

print("🔍 DETR Loaded Successfully")

# COCO weapon-like classes
violent_objects = ["knife", "baseball bat", "sports ball"]

weapon_frame_count = 0
all_detected_objects = []

# Preview Video
print("\n🎬 TEST VIDEO PREVIEW")
display(Video(VIDEO_PATH, embed=True))

# ---------------- VIDEO PROCESSING ----------------
cap = cv2.VideoCapture(VIDEO_PATH)
frame_count = 0

with torch.no_grad():
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % FRAME_INTERVAL == 0:

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame_rgb)

            detr_inputs = detr_processor(images=image, return_tensors="pt")
            detr_inputs = {k: v.to(DEVICE) for k, v in detr_inputs.items()}

            outputs = detr_model(**detr_inputs)

            target_sizes = torch.tensor([image.size[::-1]]).to(DEVICE)

            results = detr_processor.post_process_object_detection(
                outputs,
                target_sizes=target_sizes,
                threshold=DETR_THRESHOLD
            )[0]

            frame_has_weapon = False

            for score, label, box in zip(
                results["scores"],
                results["labels"],
                results["boxes"]
            ):
                class_name = detr_model.config.id2label[label.item()]
                all_detected_objects.append(class_name)

                if class_name in violent_objects:
                    frame_has_weapon = True

            if frame_has_weapon:
                weapon_frame_count += 1

        frame_count += 1

cap.release()

gc.collect()
torch.cuda.empty_cache()

# ---------------- FINAL DECISION ----------------

unique_objects = list(set(all_detected_objects))

if weapon_frame_count >= MIN_WEAPON_FRAMES:
    final_decision = "🚨 VIOLENCE DETECTED (Weapon Present)"
else:
    final_decision = "✅ NON-VIOLENCE"

print("\n🔎 Detected Objects:", unique_objects)
print("🔢 Weapon Appeared In Frames:", weapon_frame_count)
print("\n🏁 FINAL DECISION:", final_decision)

## Scrath DETR

In [1]:
import torch
from datasets import load_dataset
from transformers import (
    DetrImageProcessor,
    DetrForObjectDetection,
    TrainingArguments,
    Trainer
)
import os

In [2]:
# ================= CONFIG =================
# ================= CONFIG =================
DATASET_PATH = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Data/Video/combined_gunsnknifes"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 2
OUTPUT_DIR = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Model/detr_weapon_model"
# ==========================================

print("Using device:", DEVICE)

Using device: cuda


In [3]:
# Load processor
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")

# Load model
model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)

model.to(DEVICE)

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |                                                                                        
---------------------------------------------------------------+------------+----------------------------------------------------------------------------------------
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                            

DetrForObjectDetection(
  (model): DetrModel(
    (backbone): DetrConvEncoder(
      (model): FeatureListNet(
        (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (bn1): DetrFrozenBatchNorm2d()
        (act1): ReLU(inplace=True)
        (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
        (layer1): Sequential(
          (0): Bottleneck(
            (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn1): DetrFrozenBatchNorm2d()
            (act1): ReLU(inplace=True)
            (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (bn2): DetrFrozenBatchNorm2d()
            (drop_block): Identity()
            (act2): ReLU(inplace=True)
            (aa): Identity()
            (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn3): DetrFrozenBatchNorm2d()
            (act3): ReLU(inplace

In [4]:
# Load COCO dataset
dataset = load_dataset(
    "coco",
    data_dir=DATASET_PATH,
)

print(dataset)


DatasetNotFoundError: Dataset 'coco' doesn't exist on the Hub or cannot be accessed.

In [ ]:
# Transform function
def transform(example):
    images = example["image"]
    annotations = example["annotations"]

    encoding = processor(
        images=images,
        annotations=annotations,
        return_tensors="pt"
    )

    return encoding

dataset = dataset.with_transform(transform)


In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=30,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    evaluation_strategy="steps",
    save_total_limit=2,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
)


In [ ]:
trainer.train()

In [ ]:
# Save final model
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print("Training complete. Model saved.")

## Correct Scratch

In [1]:
import os
import torch
import json
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import DetrImageProcessor, DetrForObjectDetection
from torch.optim import AdamW
from tqdm import tqdm

In [2]:
# ================= CONFIG =================
DATASET_PATH = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Data/Video/combined_gunsnknifes"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 2
EPOCHS = 5
BATCH_SIZE = 2  # keep small for 4GB GPU
OUTPUT_DIR = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Model/detr_weapon_model"
# ==========================================

print("Using device:", DEVICE)

Using device: cuda


In [3]:
# Load processor and model
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)

model.to(DEVICE)

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |                                                                                        
---------------------------------------------------------------+------------+----------------------------------------------------------------------------------------
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                            

DetrForObjectDetection(
  (model): DetrModel(
    (backbone): DetrConvEncoder(
      (model): FeatureListNet(
        (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (bn1): DetrFrozenBatchNorm2d()
        (act1): ReLU(inplace=True)
        (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
        (layer1): Sequential(
          (0): Bottleneck(
            (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn1): DetrFrozenBatchNorm2d()
            (act1): ReLU(inplace=True)
            (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (bn2): DetrFrozenBatchNorm2d()
            (drop_block): Identity()
            (act2): ReLU(inplace=True)
            (aa): Identity()
            (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn3): DetrFrozenBatchNorm2d()
            (act3): ReLU(inplace

In [4]:
# ---------------- Custom Dataset ----------------

class COCODataset(Dataset):
    def __init__(self, root, split):
        self.root = root
        self.split = split
        
        ann_path = os.path.join(root, split, "_annotations.coco.json")
        with open(ann_path) as f:
            self.coco = json.load(f)
        
        self.images = self.coco["images"]
        self.annotations = self.coco["annotations"]

        # group annotations by image_id
        self.ann_by_image = {}
        for ann in self.annotations:
            img_id = ann["image_id"]
            if img_id not in self.ann_by_image:
                self.ann_by_image[img_id] = []
            self.ann_by_image[img_id].append(ann)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info["id"]

        img_path = os.path.join(
            self.root,
            self.split,
            "images",
            img_info["file_name"]
        )

        image = Image.open(img_path).convert("RGB")

        anns = self.ann_by_image.get(img_id, [])

        target = {
            "image_id": img_id,
            "annotations": anns
        }

        encoding = processor(
            images=image,
            annotations=target,
            return_tensors="pt"
        )

        pixel_values = encoding["pixel_values"].squeeze()
        labels = encoding["labels"][0]

        return pixel_values, labels


In [5]:
# Load datasets
train_dataset = COCODataset(DATASET_PATH, "train")
val_dataset = COCODataset(DATASET_PATH, "val")

def collate_fn(batch):
    pixel_values = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    encoding = processor.pad(
        pixel_values,
        return_tensors="pt"
    )

    return {
        "pixel_values": encoding["pixel_values"],
        "pixel_mask": encoding["pixel_mask"],
        "labels": labels
    }

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn
)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

In [6]:
# ---------------- Training Loop ----------------

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch in loop:
       pixel_values = batch["pixel_values"].to(DEVICE)
       pixel_mask = batch["pixel_mask"].to(DEVICE)
       labels = [{k: v.to(DEVICE) for k, v in t.items()} for t in batch["labels"]]

       outputs = model(
        pixel_values=pixel_values,
        pixel_mask=pixel_mask,
        labels=labels)
       
       loss = outputs.loss

       optimizer.zero_grad()
       loss.backward()
       optimizer.step()

       total_loss += loss.item()
       loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader)}")



Epoch 1/5:   1%|          | 23/3644 [00:12<33:13,  1.82it/s, loss=1.19]


OutOfMemoryError: CUDA out of memory. Tried to allocate 24.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 8.31 MiB is free. Process 2573 has 71.68 MiB memory in use. Including non-PyTorch memory, this process has 3.57 GiB memory in use. Of the allocated memory 3.38 GiB is allocated by PyTorch, and 83.15 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Save model
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print("Training complete. Model saved to", OUTPUT_DIR)